In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [ ]:
path = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/02_extracting_T_cells_and_clustering/results_drop_10-12-5-8-9__20260111_164341/02_adata_T_reclustered_after_drop.h5ad"

In [ ]:
adata = sc.read_h5ad(path)
print(adata)



In [ ]:
keep = {str(i) for i in range(7)}
ad_0_5 = adata[adata.obs['leiden_T_0.6'].astype(str).isin(keep)].copy()

In [ ]:
ad_0_5.X = np.log1p(ad_0_5.layers['norm_expr'])

In [ ]:
import pandas as pd

ad = ad_0_5
old_key = "leiden_T_0.6"
new_key = "leiden_T_0.6_relabel"

# 映射：old -> new
mapping = {
    "4": "1",
    "0": "2",
    "1": "3",
    "2": "4",
    "3": "5",
    "5": "6",
    "6": "7",
}

# 确保是字符串（scanpy 的 leiden 通常本来就是字符串）
ad.obs[old_key] = ad.obs[old_key].astype(str)

# 生成新标签；未在 mapping 里的（比如 7/8）会变成 NaN
ad.obs[new_key] = ad.obs[old_key].map(mapping)

# 可选：把没映射到的保留原值（例如 7/8），避免 NaN
ad.obs[new_key] = ad.obs[new_key].fillna(ad.obs[old_key])

# 设为有序类别（按你定义的顺序 1..7，再加上可能的 7/8）
ordered = ["1","2","3","4","5","6","7"]
extras = [c for c in sorted(ad.obs[new_key].unique()) if c not in ordered]
ad.obs[new_key] = pd.Categorical(ad.obs[new_key], categories=ordered + extras, ordered=True)

In [ ]:
import scanpy as sc
import numpy as np

ad = ad_0_5

# -----------------------------
# 1) Genes picked for each cluster (from your DE-based selection)
# -----------------------------
genes = [
    # c1 Effector
    "GZMK", "NKG7", "CST7", "CTSW", "CCL4",
    # c2 Exhausted
    "ITGAE", "ENTPD1", "CD96", "ZEB2", "ENTPD1-AS1",
    # c0 Intermediate
    "CCL5", "IFNG", "GZMA", "GZMH", "CD8A", "TOX",
    # c3 Treg
    "CTLA4", "TNFRSF18", "TNFRSF4", "BATF", "MAF", "IL2RA", "FOXP3",
    # c4 Naive CD8 (naive/early activation-like)
    "IL7R", "ANXA1", "FOS", "JUN", "ZFP36", "NFKBIA", "DUSP1",
    # c5 Progenitor / central-memory-like CD4
    "TCF7", "LTB", "IL6ST", "AFF3", "TNFSF8",
    # c6 Cycling
    "MKI67", "TYMS", "STMN1", "PCLAF", "TUBA1B",
]

# Keep only genes present in the object
genes_present = [g for g in genes if g in ad.var_names or (ad.raw is not None and g in ad.raw.var_names)]
missing = sorted(set(genes) - set(genes_present))
print(f"[INFO] genes requested: {len(genes)} | present: {len(genes_present)} | missing: {len(missing)}")
if missing:
    print("[WARN] missing genes (not in var_names/raw.var_names):", ", ".join(missing))

# -----------------------------
# 2) Make sure cluster label is categorical and ordered (0..8)
# -----------------------------
cluster_key = "leiden_T_0.6_relabel"
ad.obs[cluster_key] = ad.obs[cluster_key].astype("category")

# order clusters numerically if possible
try:
    cats = sorted(ad.obs[cluster_key].cat.categories, key=lambda x: int(x))
    ad.obs[cluster_key] = ad.obs[cluster_key].cat.reorder_categories(cats, ordered=True)
except Exception:
    pass

# -----------------------------
# 3) Choose data source: layer/raw/X
#    - Prefer layer 'norm_expr_log1p' if exists (common in your pipeline)
# -----------------------------
use_layer = None
if "norm_expr_log1p" in ad.layers:
    use_layer = "norm_expr_log1p"
    print("[INFO] Using ad.layers['norm_expr_log1p'] for plotting.")
elif ad.raw is not None:
    print("[INFO] Using ad.raw for plotting.")
else:
    print("[INFO] Using ad.X for plotting (ensure it's normalized/log1p).")

# -----------------------------
# 4) Seurat-like heatmap options
#    A) matrixplot: cluster-mean expression (compact, Seurat-ish)
#    B) heatmap: per-cell heatmap ordered by cluster (bigger figure)
# -----------------------------
sc.settings.figdir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells"
# A) Cluster-average heatmap (recommended for paper-style Seurat-like panel)
sc.pl.matrixplot(
    ad,
    var_names=genes_present,
    groupby=cluster_key,
    #layer=use_layer,
    #use_raw=(ad.raw is not None and use_layer is None),
    standard_scale="var",
    cmap="viridis",
    dendrogram=False,
    figsize=(11, 4.5),
    show=False,
    save="_Tcell_markers_matrixplot_leiden_T_0.6.png",
)

# B) Per-cell heatmap (can be heavy; uncomment if you want the Seurat full heatmap look)
sc.pl.heatmap(
     ad,
     var_names=genes_present,
     groupby=cluster_key,
     layer=use_layer,
     use_raw=(ad.raw is not None and use_layer is None),
     standard_scale="var",
     cmap="viridis",
     dendrogram=False,
     swap_axes=True,      # genes on y, clusters on x (often nicer)
     show_gene_labels=True,
     figsize=(12, 7),
)


In [ ]:
sc.settings.figdir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells"
# A) Cluster-average heatmap (recommended for paper-style Seurat-like panel)
sc.pl.matrixplot(
    ad,
    var_names=genes_present,
    groupby=cluster_key,
    #layer=use_layer,
    #use_raw=(ad.raw is not None and use_layer is None),
    standard_scale="var",
    cmap="viridis",
    dendrogram=False,
    figsize=(11, 4.5),
    show=False,
    save="_Tcell_markers_matrixplot_leiden_T_0.6.png",
)

In [ ]:
genes_present  = [
    # Naive CD8 anchors (c4)
    "IL7R","ANXA1","FOS","JUN","ZFP36","NFKBIA","DUSP1",
    
    # Transitional/activation helpers (c0)
    "RGS1","THEMIS","TOX",
    # Effector module (c1 strong; c0 shared)
    "GZMK","NKG7","CST7","CTSW","CCL4",
    "GZMA","GZMH","LYST",

    # Exhaustion/resident-like module (c2 strong; c0 shared)
    "ITGAE","ENTPD1","CD96","ENTPD1-AS1","ZEB2","CD160",
    "CMIP","CAMK4","PRKCH",

    # Treg anchors (c3)
    "CTLA4","ICOS","TNFRSF18","TNFRSF4","BATF","MAF","FOXP3","IL2RA",

    # Progenitor/TCM-like CD4 anchors (c5)
    "TCF7","LTB","IL6ST","AFF3","TNFSF8",

    # Cycling anchors (c6)
    "MKI67","TYMS","STMN1","PCLAF","RRM2","HMGB2",
]


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.matrixplot(
    ad,
    var_names=genes_present,
    groupby=cluster_key,
    layer=use_layer,
    use_raw=(ad.raw is not None and use_layer is None),
    standard_scale="var",
    cmap="viridis",
    dendrogram=False,
    swap_axes=True,          # <- flip x/y
    figsize=(8, 10),         # <- often better after swapping
    show=False,
)

fig = plt.gcf()
axes = fig.get_axes()

# find an axis that has BOTH x and y ticklabels (usually the main heatmap axis)
main_ax = None
for ax in axes:
    if len(ax.get_xticklabels()) > 0 and len(ax.get_yticklabels()) > 0:
        main_ax = ax
        break
if main_ax is None:
    main_ax = axes[0]

# Now x-axis are clusters, y-axis are genes
main_ax.tick_params(axis="x", labelsize=14)   # cluster labels
main_ax.tick_params(axis="y", labelsize=12)   # gene labels
plt.setp(main_ax.get_xticklabels(), rotation=0, ha="center")   # clusters often no rotation
plt.setp(main_ax.get_yticklabels(), rotation=0, ha="right") 

# Save with high dpi
fig.savefig("/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells/Tcell_markers_matrixplot_leiden_T_0.6.png", dpi=400, bbox_inches="tight")

plt.close(fig)
print("Saved PNG (dpi=400) and PDF.")

In [ ]:
sc.pl.umap(
    ad,
    color=[new_key],
    legend_loc="on data",   # 或 'right margin'
    frameon=False,
    size=8,
    title="",
    show=False,
)
ax = plt.gca()

# 给“on data”的cluster数字标号加白底留白（遮住下面点）
for t in ax.texts:
    if t.get_text().strip().isdigit():  # 只处理数字标号
        t.set_color("black")            # 黑字更清楚（你也可改 white）
        t.set_fontweight("bold")
        t.set_fontsize(12)
        t.set_bbox(dict(
            facecolor="white",
            edgecolor="none",
            boxstyle="round,pad=0.1",  # pad 越大留白越多
            alpha=1.0
        ))

plt.savefig("/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells/UMAP_Tcell_relabel_dpi400.png", dpi=400, bbox_inches="tight")
plt.close()


In [ ]:
# -------------------------
GENESETS = {
    "score_Cytotoxic": [
    "PRF1", "IFNG", "GNLY", "NKG7",
    "GZMA", "GZMB", "GZMH", "GZMK", "GZMM",
    "KLRK1", "KLRB1", "KLRD1",
    "FCGR3A", "FGFBP2", "ZEB2",
    "CTSW", "CST7",
    ],
    "score_Exhaustion": [
            "PDCD1","CTLA4","LAG3","HAVCR2","TIGIT","TOX","TOX2","CXCL13","ENTPD1","LAYN","IKZF2"

    ],
    "score_Cycling": [
    "CCR7", "LEF1", "TCF7", "IL7R",
    "LTB", "MAL", "NOSIP",
    "SELL", "LDHB", "ACTN1",
    ],
    "score_naive": [
        "TCF7", "LEF1", "CCR7", "IL7R", "LTB", "MAL", "SELL",
        "TRAC", "CD3D", "CD3E"
    ],
    "score_Treg": [
        "FOXP3","IL2RA","IKZF2","CTLA4","LRRC32","TNFRSF18","CCR8","LAYN"
    ]
}

In [ ]:


# -------------------------
# 3) 打分函数：自动取交集，避免缺基因报错
# -------------------------
var_set = set(ad_0_5.var_names)

def add_score(obs_key: str, genes: list[str]):
    genes_use = [g for g in genes if g in var_set]
    if len(genes_use) < 2:
        print(f"[WARN] {obs_key}: only {len(genes_use)} genes found -> skip. Found: {genes_use}")
        return
    sc.tl.score_genes(
        ad_0_5,
        gene_list=genes_use,
        score_name=obs_key,
        use_raw=False,
        random_state=0,
    )
    print(f"{obs_key}: used {len(genes_use)}/{len(genes)} genes")

for k, gs in GENESETS.items():
    add_score(k, gs)


In [ ]:
outdir = '/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells/scores_violin'

In [ ]:
ad_0_5_subset = ad_0_5[ad_0_5.obs["leiden_T_0.6_relabel"].isin(["1", "2", "3", "4"])].copy()
for g in ["score_Cytotoxic", "score_Exhaustion", "score_Cycling", "score_naive", "score_Treg"]:
    sc.pl.violin(
        ad_0_5,
        [g],
        groupby="leiden_T_0.6_relabel",
        rotation=0,
        stripplot=True,
        jitter=0.25,
        show=False,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"violin_{g}_by_cluster.png"), dpi=200)
    plt.close()

In [ ]:
genes_to_plot = [
    "CD8A", "CD8B",   # CD8
    "CD4",            # CD4
    "TRAC", "CD3D",   # T cell
    "FOXP3", "IL2RA"  # Treg（可选，但很好用）
]

# 只保留数据里存在的基因，避免报错
genes_to_plot = [g for g in genes_to_plot if g in ad_0_5.var_names]
print("Genes found:", genes_to_plot)

In [ ]:
outdir = '/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/T_cells/CD8_vs_CD4'

In [ ]:
import os
for g in genes_to_plot:
    sc.pl.violin(
        ad_0_5,
        keys=g,
        groupby="leiden_T_0.6_relabel",
        stripplot=True,
        jitter=0.25,
        rotation=0,
        show=False,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"violin_{g}_by_cluster.png"), dpi=200)
    plt.close()

print("Saved to:", outdir)

In [ ]:
print(ad_0_5)
print(ad_0_5.obs['Respond'])

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# inputs
# -----------------------
ad = ad_0_5  # <- 你的 AnnData
cluster_key = "leiden_T_0.6_relabel"   # <- 你的cluster列名
group_key = "Respond"                  # <- "Responder"/"Non-Responder"
out_dir = "./tcell_cluster_response_counts"
os.makedirs(out_dir, exist_ok=True)

# -----------------------
# counts table (rows=cluster, cols=Respond)
# -----------------------
ct = pd.crosstab(ad.obs[cluster_key], ad.obs[group_key])

# 可选：按数字顺序排序（如果cluster是"1","2"...字符串）
try:
    ct = ct.loc[sorted(ct.index, key=lambda x: int(x))]
except Exception:
    pass

# 确保列顺序
cols = [c for c in ["Responder", "Non-Responder"] if c in ct.columns]
ct = ct[cols]

ct.to_csv(os.path.join(out_dir, "counts_cluster_by_respond.csv"))

# -----------------------
# plot: stacked counts barplot
# -----------------------
ax = ct.plot(kind="bar", stacked=True, figsize=(7, 4.5))

ax.set_xlabel("T-cell cluster")
ax.set_ylabel("Number of cells")
ax.set_title("")

# 关键：把 x 轴标签转回水平（0度）
ax.tick_params(axis="x", labelrotation=0)

# 如果还想更大字体
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

ax.legend(title=group_key, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "bar_counts_within_cluster_stacked.png"),
            dpi=400, bbox_inches="tight")
plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# inputs
# -----------------------
ad = ad_0_5
cluster_key = "leiden_T_0.6_relabel"
group_key = "Respond"  # "Responder"/"Non-Responder"
out_dir = "./tcell_cluster_response_counts"
os.makedirs(out_dir, exist_ok=True)

# -----------------------
# counts table
# -----------------------
ct = pd.crosstab(ad.obs[cluster_key], ad.obs[group_key])

# 可选：按数字顺序排序
try:
    ct = ct.loc[sorted(ct.index, key=lambda x: int(x))]
except Exception:
    pass

# 确保列顺序
cols = [c for c in ["Responder", "Non-Responder"] if c in ct.columns]
ct = ct[cols]

ct.to_csv(os.path.join(out_dir, "counts_cluster_by_respond.csv"))

# -----------------------
# convert to percentages (each bar same height)
# -----------------------
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.to_csv(os.path.join(out_dir, "percent_cluster_by_respond.csv"))

# -----------------------
# plot: stacked percent barplot
# -----------------------
ax = ct_pct.plot(kind="bar", stacked=True, figsize=(8, 4.5), width=0.95)


ax.set_xlabel("T-cell cluster")
ax.set_ylabel("Percent of cells (%)")
ax.set_title("")

ax.set_ylim(0, 100)  # 固定 0-100%，保证每个bar同高度
ax.tick_params(axis="x", labelrotation=0)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

ax.legend(title=group_key, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "bar_percent_within_cluster_stacked.png"),
            dpi=400, bbox_inches="tight")
plt.show()


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

ad = ad_0_5
cluster_key = "leiden_T_0.6_relabel"
group_key = "Respond"   # "Responder" / "Non-Responder"
clusters_to_plot = ["4", "3"]

genes = ["CTSW","JUN","KLF2","HSP90AA1","HSPA1A","HSPA1B","GZMH","NKG7"]
use_layer = None  # e.g. "norm_expr_log1p" if you want

ad.obs[group_key] = ad.obs[group_key].astype("category")
desired_order = ["Responder", "Non-Responder"]
ad.obs[group_key] = ad.obs[group_key].cat.reorder_categories(
    [x for x in desired_order if x in ad.obs[group_key].cat.categories],
    ordered=True
)

# New colors (not your cluster palette; not blue/orange)
respond_palette = {"Responder": "#6a3d9a", "Non-Responder": "#1f78b4"}  # purple / deep blue

# --- make figure narrower: shrink X by ~1/3 ---
BASE_W, BASE_H = 4.0, 4.2
W = BASE_W * (2/3)   # <- shrink by 1/3
H = BASE_H

for cl in clusters_to_plot:
    ad_cl = ad[ad.obs[cluster_key].astype(str) == str(cl)].copy()

    for g in genes:
        if g not in ad_cl.var_names:
            print(f"[INFO] cluster {cl}: missing gene {g} (skipped)")
            continue

        sc.pl.violin(
            ad_cl,
            keys=g,
            groupby=group_key,
            layer=use_layer,
            use_raw=(ad_cl.raw is not None and use_layer is None),
            stripplot=True,
            jitter=0.25,
            rotation=0,
            show=False,
        )

        fig = plt.gcf()
        fig.set_size_inches(W, H)  # <- narrower

        for ax in fig.axes:
            ax.tick_params(axis="x", labelsize=8)
            ax.tick_params(axis="y", labelsize=8)
            ax.set_xlabel("")
            ax.set_ylabel(g, fontsize=8)

            # legend outside so it doesn't eat width
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()  # <- remove legend entirely (you already know colors)

        plt.tight_layout()
        out_png = f"violin_{g}__NR_vs_R__cluster{cl}.png"
        plt.savefig(out_png, dpi=400, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK] Saved: {out_png}")
